In [ ]:
from Datasets.wiki_people.wiki_people import WikiPeople
from Datasets.utilities import get_person_nationality_with_dbpedia_link, get_country, country_name_standardizer

import os
os.environ['NationalityDatasets'] = "./data/"
wiki_people = WikiPeople()

### Reading the raw data

In [ ]:
data = wiki_people.get_raw_data()
data.shape

### Exploring the data

In [ ]:
data.head(10)

##### Checking None values

In [ ]:
data.fillna('', inplace=True)

In [ ]:
data[data['name']=='']

In [ ]:
# data[data['birth place']=='']
data.head()

In [ ]:
print(data.shape)
data = data[(data['name'] != '') & (data['birth place'] != '')]
print(data.shape)

##### Checking valid values

In [ ]:
data['birth place'].value_counts()

In [ ]:
data['country'] = data['birth place'].apply(lambda x: x.split(',')[-1].replace('_',' ').strip())
data['country'].value_counts()

In [ ]:
data['standard_country'] = data['country'].apply(lambda x: country_name_standardizer(x))

In [ ]:
from Datasets.mapping import nationality_to_country

data['standard_country'] = data.apply(lambda x: nationality_to_country(x['country']) if x.standard_country == '' else x.standard_country, axis=1)

In [ ]:
mapping = {
    'Wales': 'United Kingdom',
    'New South Wales': 'United Kingdom',
    'Welsh people': 'United Kingdom',
    'Scotland': 'United Kingdom',
    'London': 'United Kingdom',
    'UK': 'United Kingdom',
    'England': 'United Kingdom',
    'British people': 'United Kingdom',
    'English people': 'United Kingdom',
    'Kingdom of England': 'United Kingdom',
    'Northern Ireland': 'Ireland',
    'Irish people': 'Ireland',
    'Dublin': 'Ireland',
    'Berlin': 'Germany',
    'Vienna': 'Austria',
}
data['standard_country'] = data.apply(lambda x: mapping[x['country']] if x['country'] in mapping else x['standard_country'], axis=1)

In [ ]:
# data['standard_country'] = data.apply(lambda x: get_country(x['country']) if x['standard_country'] == '' else x['standard_country'], axis=1)

In [ ]:
data[(data.standard_country=='') & (data.country!='')].country.value_counts()

In [ ]:
data.country = data.standard_country


In [ ]:
print(data.shape)
data = data[data.country!= '']
print(data.shape)

In [ ]:
data.name.value_counts()

In [ ]:
# data['dbpedia'] = data.URI.apply(lambda x: x[1:-1])
# data.dbpedia.sample(10)

In [ ]:
# data['birth place'] = data.dbpedia.apply(lambda x: get_person_nationality_with_dbpedia_link(x).split('/')[-1])
# data['birth place']

In [ ]:
# data['country'] = data['birth place'].apply(lambda x: get_country(x))
# data['country']

### Saving the data

In [ ]:
data['id'] = data.apply(lambda x:'wiki-people-'+str(x.name), axis=1)


In [ ]:
data.head()

In [ ]:
data['full-name'] = data.name

In [ ]:
processed_data = data[['id', 'full-name', 'country']]
processed_data.set_index('id', inplace=True)

In [ ]:
wiki_people.save_data(processed_data)

In [ ]:
wiki_people.data = processed_data

In [ ]:
wiki_people.save_split_data()

In [ ]:
data.to_csv('data/raw_data.csv')